# 03 - Model Development

This notebook focuses on building and evaluating predictive models that form the core of a dynamic, risk-based pricing system, as outlined in Task 4.

Modeling Goals:
Claim Probability Prediction (Classification): Predict the likelihood of a policy having a claim (HasClaim).

Target Variable: HasClaim (binary: 0 or 1)

Evaluation Metrics: Accuracy, Precision, Recall, F1-Score, ROC AUC.

Claim Severity Prediction (Regression): For policies that do have a claim, predict the TotalClaims amount.

Target Variable: TotalClaims (numerical, conditional on HasClaim > 0)

Evaluation Metrics: Root Mean Squared Error (RMSE), R-squared (R2).

Modeling Techniques to Implement and Evaluate:
Classification: RandomForestClassifier, XGBClassifier

Regression: LinearRegression, RandomForestRegressor, XGBRegressor

Pipeline Steps:
Data Preparation: Load raw data and prepare it for both classification and regression tasks using src/data_preparation.py.

Model Building: Initialize and train models using functions from src/models.py.

Model Evaluation: Evaluate each model's performance using appropriate metrics defined in src/models.py.

Feature Importance & Interpretability: Analyze feature importance using SHAP for the best-performing models.

Model Saving: Save trained models and preprocessors for future use.

This notebook will largely call the functions developed in scripts/training_models.py to ensure consistency and reusability of the pipeline.

In [3]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- Setup and Imports ---
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # For loading preprocessors and models if needed

# Dynamically determine the project root more robustly.
# This logic will find the directory that contains 'src' regardless of
# where the Jupyter kernel was launched from, assuming a standard project structure.

# Start from the current working directory (where the Jupyter kernel is running)
current_path = os.getcwd()
actual_project_root = None

# Walk up the directory tree until 'src' directory is found
while True:
    # Look for a marker file/directory, e.g., 'src' or 'scripts'
    if os.path.exists(os.path.join(current_path, 'src')) and os.path.exists(os.path.join(current_path, 'scripts')):
        actual_project_root = current_path
        break
    parent_path = os.path.dirname(current_path)
    if parent_path == current_path: # Reached file system root
        break
    current_path = parent_path

if actual_project_root is None:
    raise RuntimeError("Could not find the project root containing both 'src' and 'scripts' directories. "
                       "Please ensure your notebook is run from within the project structure.")

# Ensure the actual project root is at the very beginning of sys.path
# This ensures that 'src' and 'scripts' can be imported correctly as packages.
if actual_project_root in sys.path:
    sys.path.remove(actual_project_root) # Remove if already present to ensure it's at index 0
sys.path.insert(0, actual_project_root)

print(f"Detected project root: '{actual_project_root}'")
print(f"Ensured project root is at the beginning of sys.path.")
print(f"Current sys.path (first 3 entries): {sys.path[:3]}") # Print to confirm

# Import main training function from scripts
from scripts.training_models import main as run_training_pipeline

# Import relevant config constants (for reference, the training_models script uses them directly)
from src.config import (
    RAW_DATA_PATH, MODEL_DIR,
    NUMERICAL_FEATURES_CLASSIFICATION, CATEGORICAL_FEATURES_CLASSIFICATION, DATE_FEATURES_CLASSIFICATION, COLS_TO_DROP_CLASSIFICATION,
    NUMERICAL_FEATURES_REGRESSION, CATEGORICAL_FEATURES_REGRESSION, DATE_FEATURES_REGRESSION, COLS_TO_DROP_REGRESSION
)

# Configure plotting styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


Detected project root: 'C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis'
Ensured project root is at the beginning of sys.path.
Current sys.path (first 3 entries): ['C:\\Users\\benke\\Desktop\\DataScience\\10Academy\\Kefiya_AI_Mastery\\Week3\\KAIM_W3_Insurance_risk_analysis\\KAIM_W03_AlphaCare_Insurance_Risk_Analysis', 'C:\\Users\\benke\\Desktop\\DataScience\\10Academy\\Kefiya_AI_Mastery\\Week3\\KAIM_W3_Insurance_risk_analysis\\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\\notebooks', 'C:\\Users\\benke\\anaconda3\\python311.zip']
Project root 'C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis' was already in sys.path.


# 1. Execute the Model Training Pipeline
The core logic for data preparation, model training, evaluation, and SHAP analysis is encapsulated in scripts/training_models.py. We will call its main() function directly from this notebook to execute the entire pipeline.

This step will:

Load raw data.

Prepare data for both classification and regression tasks.

Train RandomForest and XGBoost models for classification.

Train Linear Regression, RandomForest, and XGBoost models for regression.

Evaluate all models and print metrics.

Generate SHAP plots for XGBoost models.

Save all trained models and preprocessors to the models/ directory.

In [ ]:
print("Running the full model training pipeline from scripts/training_models.py...")
run_training_pipeline()
print("\nModel training pipeline execution complete.")

Running the full model training pipeline from scripts/training_models.py...

--- Starting Model Training Pipeline ---


C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\src\data_preparation.py:28: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter='|')


Raw data loaded successfully from C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\data\raw\MachineLearningRating_v3.txt. Shape: (1000098, 52)

--- Preparing data for Claim Probability (Classification) ---
Initial numeric coercion (empty string/whitespace to NaN, then to_numeric) complete.
Dropped initial columns: ['PolicyID'].


C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\src\feature_engineering.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_engineered[col] = pd.to_datetime(df_engineered[col], errors='coerce')


Feature engineering complete using src/feature_engineering.py.
Prepared data for Classification: Target 'HasClaim'.
X filtered to only include 54 intended features.
Final missing data handling (imputation) complete.

--- Handling High Cardinality Categorical Features (max_categories=50) ---
  Column 'AccountType' has 3 unique categories (no reduction needed).
  Column 'AlarmImmobiliser' has 2 unique categories (no reduction needed).
  Column 'Bank' has 11 unique categories (no reduction needed).
  Column 'Citizenship' has 3 unique categories (no reduction needed).
  Column 'Converted' has 2 unique categories (no reduction needed).
  Column 'Country' has 1 unique categories (no reduction needed).
  Column 'CoverCategory' has 27 unique categories (no reduction needed).
  Column 'CoverGroup' has 14 unique categories (no reduction needed).
  Column 'CoverType' has 21 unique categories (no reduction needed).
  Column 'CrossBorder' has 1 unique categories (no reduction needed).
  Column 'Exc

# 2. Review Model Performance and Interpretability
The previous step printed detailed evaluation metrics and generated SHAP plots. Here, we will recap the key findings and discuss their business implications. You can refer to the output of the cell above for the precise metrics and SHAP plots.

2.1. Claim Probability Prediction (Classification Models)
RandomForestClassifier vs. XGBClassifier:

Metrics Comparison: (Refer to the output above for exact values)

Accuracy: How many predictions were correct overall.

Precision: Of the predicted claims, how many were actual claims (minimizes false positives).

Recall: Of the actual claims, how many were correctly identified (minimizes false negatives).

F1-Score: Harmonic mean of precision and recall.

ROC AUC: Measures the model's ability to distinguish between classes, crucial for imbalanced datasets.

Typical Observations: XGBoost often performs very well on tabular data and might outperform RandomForest, especially if tuned. Given the class imbalance in HasClaim, ROC AUC is usually a more reliable metric than accuracy.

Business Implication: A higher ROC AUC indicates a better ability to rank policies by their claim probability. This is vital for a risk-based pricing system, allowing AlphaCare to differentiate between high-risk and low-risk policies effectively.

2.2. Claim Severity Prediction (Regression Models)
LinearRegression vs. RandomForestRegressor vs. XGBRegressor:

Metrics Comparison: (Refer to the output above for exact values)

RMSE: Measures the average magnitude of the errors. Lower RMSE is better.

R-squared (R2): Represents the proportion of variance in the dependent variable that is predictable from the independent variables. Higher R2 is better.

Typical Observations: Tree-based models (RandomForest and XGBoost) are generally more robust to non-linear relationships and outliers, often yielding lower RMSE and higher R2 compared to Linear Regression for complex real-world data. XGBoost often leads the pack in performance.

Business Implication: Accurate prediction of claim severity allows AlphaCare to set premiums that adequately cover expected claim costs. Lower RMSE means tighter predictions, reducing financial surprises and improving profitability. A high R2 indicates that the features chosen explain a significant portion of the variability in claim amounts.

2.3. Model Interpretability (SHAP Analysis)
The SHAP summary plots generated by scripts/training_models.py provide crucial insights into feature importance and their impact on predictions for the XGBoost models.

For Claim Probability XGBoost Model:

Top 5-10 Influential Features: (Examine the SHAP bar plot for Claim Probability XGBoost Model)

Identify the features with the largest absolute SHAP values (top of the bar plot).

Impact Direction: (Examine the SHAP dot plot for Claim Probability XGBoost Model)

Example Interpretation: If num__VehicleAge_at_Transaction is a top feature and its SHAP values for higher ages are red (positive), it means older vehicles tend to increase the predicted probability of a claim. Conversely, if values for num__TotalPremium are blue (negative), it could indicate that policies with higher premiums (perhaps due to better coverage or safer drivers/vehicles) have a lower predicted claim probability.

Business Interpretation: Understanding these drivers allows AlphaCare to target specific risk factors. For example, if num__VehicleAge_at_Transaction is a major driver of claim probability, this provides quantitative evidence to refine age-based premium adjustments or explore maintenance incentives for older vehicles.

For Claim Severity XGBoost Model:

Top 5-10 Influential Features: (Examine the SHAP bar plot for Claim Severity XGBoost Model)

Identify the features with the largest absolute SHAP values.

Impact Direction: (Examine the SHAP dot plot for Claim Severity XGBoost Model)

Example Interpretation: If num__SumInsured is a top feature and its SHAP values for higher insured amounts are red (positive), it means higher insured values lead to higher predicted claim severity. If certain cat__VehicleType_X features are highly influential and push predictions down (blue), it suggests those vehicle types are associated with lower claim costs.

Business Interpretation: Knowing what drives claim severity helps in setting adequate reserves and understanding potential cost drivers. If num__SumInsured is a dominant factor, it confirms that policy value strongly dictates potential payout. This can also guide product development or risk selection criteria.

# 3. Overall Conclusion and Next Steps
This model development phase has successfully:

Built and evaluated robust predictive models for both claim probability and claim severity.

Identified the best-performing models (likely XGBoost for both tasks, given its typical performance).

Provided interpretable insights into key risk drivers using SHAP values, quantifying their impact on predictions.

These models are foundational for AlphaCare's dynamic, risk-based pricing system. The equation Premium = (Predicted Probability of Claim * Predicted Claim Severity) + Expense Loading + Profit Margin can now be implemented using the outputs of these models.

Next Steps:

Deployment: Integrate the trained models (saved as .joblib files in the models/ directory) into a production environment for real-time premium calculation.

Validation: Continuously monitor model performance on new data and retrain as needed.

Refinement: Explore more advanced feature engineering, hyperparameter tuning, or alternative modeling techniques (e.g., neural networks for highly complex patterns, or generalized linear models for actuarial consistency).

Business Integration: Work closely with actuarial and business teams to define the Expense Loading and Profit Margin components, and ensure the risk-based premiums align with business objectives and regulatory requirements.